In [1]:
!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 MB 79.1 MB/s  0:00:01m0:00:0100:01


In [2]:
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
import math

In [5]:
TOPK = 50
sim_item_topk = 10
recall_item_num = 10

In [ ]:
DATA = '/kaggle/input/datasets/jiaweiwei0311/tianchinewsrec/'

train_click = pd.read_csv(DATA + 'train_click_log.csv')
testA_click = pd.read_csv(DATA + 'testA_click_log.csv')
articles = pd.read_csv(DATA + 'articles.csv').rename(columns={"article_id":"click_article_id"})
all_click = pd.concat([train_click,testA_click], ignore_index=True)
train_click.columns.tolist(),testA_click.columns.tolist(),articles.columns.tolist()
TopK_articles = all_click['click_article_id'].value_counts().index[:TOPK]


In [7]:
item_sim = defaultdict(dict) # wij
item_click = defaultdict(int) # 物品被点击次数
user_click_items_dict = all_click.groupby('user_id')['click_article_id'].agg(list).to_dict()
for user,click_items in tqdm(user_click_items_dict.items()):
    for i in click_items:
        item_click[i]+=1
        for j in click_items:
            if i==j:
                continue
            item_sim[i].setdefault(j, 0)
            item_sim[i][j] += 1/math.log(1+len(click_items))



100%|██████████| 250000/250000 [00:36<00:00, 6850.68it/s] 


In [8]:
for i, related in tqdm(item_sim.items()):
    for j in related:
        related[j]/=math.sqrt(item_click[i] * item_click[j])
# item_sim = {
#     i: {j: w / math.sqrt(item_click[i] * item_click[j]) for j, w in related.items()}
#     for i, related in item_sim.items()
# }


100%|██████████| 35197/35197 [00:05<00:00, 5941.70it/s] 


In [9]:
topk_sim = {
    i:sorted(related.items(), key=lambda x:x[1], reverse= True)[:sim_item_topk]
    for i, related in item_sim.items()
}
user_recall_items_dict = {}
user_id_list=testA_click['user_id'].unique()
for user_id in tqdm(user_id_list): # 对每个用户召回
    item_rank = defaultdict(int)
    history=user_click_items_dict[user_id]
    history_set=set(history)
    for history_item_id in history:
        for j,sim in topk_sim.get(history_item_id,[]): 
            if j in history_set:
                continue
            item_rank[j]+=sim
    if len(item_rank)<sim_item_topk:
        for rank,j in enumerate(TopK_articles):
            if j in item_rank:
                continue
            item_rank[j]=-rank-1
            if len(item_rank)==sim_item_topk:
                break
    user_recall_items_dict[user_id]=sorted(item_rank.items(), key=lambda x:x[1], reverse=True)[:recall_item_num]


100%|██████████| 50000/50000 [00:05<00:00, 9917.36it/s] 


In [19]:
# 内核机器上项目代码的目录（改成你实际上传的位置）
PROJ = '/kaggle/working/news_recommender'

import os, sys

assert os.path.isdir(PROJ), f'PROJ 不存在：{PROJ}'
if PROJ not in sys.path:
    sys.path.insert(0, PROJ)
os.chdir(PROJ)

print('项目目录:', PROJ)

项目目录: /kaggle/working/news_recommender


In [28]:
# ============================================================
# 线下评估：Recall@10 / MRR@5（ItemCF baseline）
# 严格口径：只用 train 点击；每人最后一条点击当答案，其余当历史
# 依赖上面的 bootstrap 单元格
# ============================================================
import os
import pandas as pd

from data_processing import get_all_click_df, resolve_data_path
from utils import get_user_item_time_dict, get_item_topk_click
from models.recall_baseline import itemcf_sim, generate_user_recall_dict

data_path = resolve_data_path()
cache_dir = os.environ.get('NEWS_CACHE_DIR', './cache/')

# ---- 1. 切分：每人最后一条当答案 ----
off = get_all_click_df(data_path=data_path, offline=True)
off = off.sort_values(['user_id', 'click_timestamp'])
val_df = off.groupby('user_id').tail(1)      # 答案
trn_df = off.drop(val_df.index)              # 历史
print(f'总点击 {len(off)} | 历史 {len(trn_df)} | 答案 {len(val_df)} | 用户 {val_df.user_id.nunique()}')

# ---- 2. 只用 trn 构建历史/热门/相似度（用 val 就是泄露）----
uit_tr = get_user_item_time_dict(trn_df)
hot_tr = get_item_topk_click(trn_df, k=50)
i2i_tr = itemcf_sim(trn_df, save_path=os.path.join(cache_dir, 'itemcf_sim_offline.pkl'),
                    use_cache=False)

# 只有 1 次点击的用户，切完就没有历史；补空列表，走热门补全
for u in val_df['user_id'].unique():
    uit_tr.setdefault(u, [])

# ---- 3. 对 val 用户生成召回 ----
val_users_df = pd.DataFrame({'user_id': val_df['user_id'].unique()})
recall_off = generate_user_recall_dict(
    val_df=val_users_df,
    user_item_time_dict=uit_tr,
    i2i_sim=i2i_tr,
    sim_item_topk=10,
    recall_item_num=10,
    item_topk_click=hot_tr,
    save_path=os.path.join(cache_dir, 'user_recall_items_offline.pkl'),
    use_cache=False,
)

# ---- 4. 指标 ----
gt = dict(zip(val_df['user_id'], val_df['click_article_id']))
n = len(recall_off)

def recall_at_k(k):
    hit = sum(1 for u, lst in recall_off.items() if gt.get(u) in [i for i, _ in lst[:k]])
    return hit / n

def mrr_at_k(k):
    total = 0.0
    for u, lst in recall_off.items():
        for rank, (item, _) in enumerate(lst[:k], 1):
            if item == gt.get(u):
                total += 1.0 / rank
                break
    return total / n

print(f'评估用户数: {n}')
print(f'Recall@10 = {recall_at_k(10):.4f}')
print(f'MRR@5     = {mrr_at_k(5):.4f}')

总点击 1112623 | 历史 912623 | 答案 200000 | 用户 200000
[itemcf_sim_baseline] 🚧 正在计算相似度矩阵（无加权）...


100%|██████████| 200000/200000 [00:09<00:00, 20974.06it/s]


[itemcf_sim_baseline] ✅ 相似度矩阵已保存至：/kaggle/working/news_recommender/cache/itemcf_sim_offline.pkl
[generate_user_recall_dict] 🚀 正在生成用户召回列表...


100%|██████████| 200000/200000 [00:05<00:00, 34348.43it/s]


[generate_user_recall_dict] ✅ 召回列表保存成功：/kaggle/working/news_recommender/cache/user_recall_items_offline.pkl
评估用户数: 200000
Recall@10 = 0.3370
MRR@5     = 0.1366


In [33]:
# ============================================================
# 线下评估：Recall@10 / MRR@5（itemcf_recall 中的 ItemCF）
# 严格口径：只用 train 点击；每人最后一条点击当答案，其余当历史
# 依赖上面的 bootstrap 单元格
# ============================================================
import os
import pandas as pd

from data_processing import get_all_click_df, get_item_info_df, resolve_data_path
from utils import get_user_item_time_dict, get_item_topk_click, get_item_info_dict
from models.itemcf_recall import itemcf_sim, generate_itemcf_recall_dict

data_path = resolve_data_path()
cache_dir = os.environ.get('NEWS_CACHE_DIR', './cache/')

# ---- 1. 切分：每人最后一条当答案 ----
off = get_all_click_df(data_path=data_path, offline=True)
off = off.sort_values(['user_id', 'click_timestamp'])
val_df = off.groupby('user_id').tail(1)      # 答案
trn_df = off.drop(val_df.index)              # 历史
print(f'总点击 {len(off)} | 历史 {len(trn_df)} | 答案 {len(val_df)} | 用户 {val_df.user_id.nunique()}')

# ---- 2. 只用 trn 构建历史/热门/相似度（用 val 就是泄露）----
uit_tr = get_user_item_time_dict(trn_df)
hot_tr = get_item_topk_click(trn_df, k=50)
item_info_df = get_item_info_df(data_path)
_, _, item_created_time_dict = get_item_info_dict(item_info_df)
i2i_tr = itemcf_sim(
    trn_df,
    item_created_time_dict=item_created_time_dict,
    save_path=os.path.join(cache_dir, 'itemcf_recall_sim_offline.pkl'),
    use_cache=True,
)

# 只有 1 次点击的用户，切完就没有历史；补空列表，走热门补全
for u in val_df['user_id'].unique():
    uit_tr.setdefault(u, [])

# ---- 3. 对 val 用户生成召回 ----
val_users_df = pd.DataFrame({'user_id': val_df['user_id'].unique()})
recall_off = generate_itemcf_recall_dict(
    val_df=val_users_df,
    user_item_time_dict=uit_tr,
    i2i_sim=i2i_tr,
    sim_item_topk=10,
    recall_item_num=10,
    item_topk_click=hot_tr,
    item_created_time_dict=item_created_time_dict,
    emb_i2i_sim={},
    save_path=os.path.join(cache_dir, 'user_recall_itemcf_offline.pkl'),
    use_cache=False,
)

# ---- 4. 指标 ----
gt = dict(zip(val_df['user_id'], val_df['click_article_id']))
n = len(recall_off)

def recall_at_k(k):
    hit = sum(1 for u, lst in recall_off.items() if gt.get(u) in [i for i, _ in lst[:k]])
    return hit / n

def mrr_at_k(k):
    total = 0.0
    for u, lst in recall_off.items():
        for rank, (item, _) in enumerate(lst[:k], 1):
            if item == gt.get(u):
                total += 1.0 / rank
                break
    return total / n

print(f'评估用户数: {n}')
print(f'Recall@10 = {recall_at_k(10):.4f}')
print(f'MRR@5     = {mrr_at_k(5):.4f}')

总点击 1112623 | 历史 912623 | 答案 200000 | 用户 200000
[itemcf_sim] ✅ 使用缓存文件：/kaggle/working/news_recommender/cache/itemcf_recall_sim_offline.pkl
[generate_user_recall_dict_itemcf] 🚀 正在生成 ItemCF 召回列表...


100%|██████████| 200000/200000 [00:32<00:00, 6113.11it/s]


[generate_user_recall_dict_itemcf] ✅ 召回列表保存成功：/kaggle/working/news_recommender/cache/user_recall_itemcf_offline.pkl
评估用户数: 200000
Recall@10 = 0.3825
MRR@5     = 0.1553
